## Heuristic

In [1]:
import math
from pathlib import Path
import networkx as nx
import pandas as pd

In [ ]:
data = "C:\Line-Planning-Problem\data\mandl" 

#Parameters (Could read from config file but for now hardcoded)
tpenalty = 5
min_stops = 3
max_stops = 8
k_path = 3
budget = 750

min_lines = 4
max_lines = 6
fairness_threshold = 1.5 


<>:1: SyntaxWarning: invalid escape sequence '\L'
<>:1: SyntaxWarning: invalid escape sequence '\L'
C:\Users\csash\AppData\Local\Temp\ipykernel_12148\3887787887.py:1: SyntaxWarning: invalid escape sequence '\L'
  data = "C:\Line-Planning-Problem\data\mandl"


In [9]:
def load_data(data):
    data = Path(data, "basis")

    #Get edges and OD demand
    edges = pd.read_csv(
        Path(data, "Edge.giv"),
        sep=";", comment="#", header=None,
        names=["id", "u", "v", "length", "time", "upper"]
    )

    origin_demand = pd.read_csv(
        Path(data, "OD.giv"),
        sep=";", comment="#", header=None,
        names=["origin", "destination", "demand"]
    )

    graph = nx.Graph()

    #Add edges and demand to graph
    for index, row in edges.iterrows():
        graph.add_edge(int(row.u), int(row.v), weight=float(row.time))

    #Get demand as dict
    demand = {}
    for index, row in origin_demand.iterrows():

        if row.demand > 0:

            origin = int(row.origin)
            destination = int(row.destination)
            passengers = float(row.demand)

            demand[(origin, destination)] = passengers

    return graph, demand


#Generate candidate line pool
def gen_line_pool(graph):

    lines = {}
    seen = set()

    for start in graph.nodes:
        for end in graph.nodes:

            #Avoid loops
            if start >= end:
                continue

            
            try:
                #NX finds paths in increasing weight
                paths = nx.shortest_simple_paths(graph, start, end, weight="weight")

                #Only keep p paths
                for i, path in enumerate(paths):
                    if i >= k_path:
                        break

                    #Path length has to be within min and max stops
                    if min_stops <= len(path) <= max_stops:
                        #Make sure reverse paths are not duplicated
                        key = min(tuple(path), tuple(reversed(path)))

                        #Add path if not seen
                        if key not in seen:
                            seen.add(key)
                            lines[len(lines)] = path

            except nx.NetworkXNoPath:
                pass

    return lines


#Get cost of line based on edge weights
def get_line_cost(graph, line):
    cost = 0

    for start_stop, end_stop in zip(line, line[1:]):
        travel_time = graph[start_stop][end_stop]["weight"]
        cost += travel_time

    return cost

#Create low-high demand groups
def demand_groups(demand):
    values = pd.Series(list(demand.values()))
    first_quant, sec_quant = values.quantile([1/3, 2/3])

    groups = {"low": [], "medium": [], "high": []}

    for origin_demand, passenger_demand in demand.items():
        if passenger_demand <= first_quant:
            groups["low"].append(origin_demand)
        elif passenger_demand <= sec_quant:
            groups["medium"].append(origin_demand)
        else:
            groups["high"].append(origin_demand)

    return groups

#Evaluate selected lines
def evaluate(graph, selected_lines, line_pool, demand, groups, shortest_times):

    graph_passenger = nx.DiGraph()
    #which stop has which lines
    stop_lines = {}

    #loop through selected lines
    for line_id in selected_lines:
        path = line_pool[line_id]

        #Add line to stop dict
        for stop in path:
            stop_lines.setdefault(stop, []).append(line_id)

        # Travel along each line in both directions
        for start_stop, end_stop in zip(path, path[1:]):
            #Get travel time between stops
            travel_time = graph[start_stop][end_stop]["weight"]

            #Loop over transfer states - 0,1,2
            for transfers in range(3):
                #Add edges in both directions
                graph_passenger.add_edge((start_stop, line_id, transfers), (end_stop, line_id, transfers), weight=travel_time)
                graph_passenger.add_edge((end_stop, line_id, transfers), (start_stop, line_id, transfers), weight=travel_time)

    #Basically we update the transfer counts and add a penalty 
    for stop, lines in stop_lines.items():
        for current_line in lines:
            for next_line in lines:

                #Same line
                if current_line == next_line:
                    continue

                for transfers in range(3):

                    new_transfers = min(2, transfers + 1)

                    if transfers < 2:
                        penalty = tpenalty
                    else:
                        penalty = 0
                    
                    graph_passenger.add_edge((stop, current_line, transfers), (stop, next_line, new_transfers), weight=penalty)

 

    times = {}
    #Loop through orig-dest pairs
    for (origin, destination), passengers in demand.items():

        #Get all starting lines for origin
        starting_states = []
        for line_id in stop_lines.get(origin, []):
            starting_state = (origin, line_id, 0)
            starting_states.append(starting_state)

        #if no lines at origin, set time to infinity
        if not starting_states:
            times[(origin, destination)] = math.inf
            continue

        #Calculate shortest path for states
        distances = nx.multi_source_dijkstra_path_length(graph_passenger, starting_states, weight="weight")

        #Assume no path exists
        best_time = math.inf

        #Check every way to reach destination
        for line_id in stop_lines.get(destination, []):
            for transfers in range(3):
                #Save only best time
                best_time = min(best_time,distances.get((destination, line_id, transfers), math.inf))

        times[(origin, destination)] = best_time

    #
    possible_demand = 0
    total_travel_time = 0

    #Get travel time
    for origin_demand_pair, passengers in demand.items():
        travel_time = times[origin_demand_pair]

        #If passengers can reach destination, addd demand and travel time
        if math.isfinite(travel_time):
            possible_demand += passengers
            total_travel_time += passengers * travel_time

    #Calculate average travel time for served passengers
    if possible_demand > 0:
        average_served_time = total_travel_time / possible_demand
    else:
        average_served_time = math.inf

    #Check whether each group is served fairly
    group_scores = {}

    for group_name, origin_demand_pairs in groups.items():

        #If group empty, then no score
        if len(origin_demand_pairs) == 0:
            continue

        #Check if everyone can travel
        has_invalid_pair = False
        for pair in origin_demand_pairs:
            if not math.isfinite(times[pair]):
                has_invalid_pair = True
                break

        if has_invalid_pair:
            group_scores[group_name] = math.inf
            continue

        #
        group_demand = 0
        group_relative_time = 0

        #Calculate weighted relative travel time for each demand group
        for pair in origin_demand_pairs:
            passengers = demand[pair]
            travel_time = times[pair]
            shortest_time = shortest_times[pair]

            relative_time = travel_time / shortest_time

            group_demand += passengers
            group_relative_time += passengers * relative_time

        group_scores[group_name] = group_relative_time / group_demand

    inequity_score = 0
    #If any group exceeds fairness boundary, add to inequity score
    for score in group_scores.values():
        if score > fairness_threshold:
            inequity_score += score - fairness_threshold

    fair = inequity_score == 0

    return (possible_demand, average_served_time, fair, inequity_score)


#Heuristic function that uses greedy construction and local search
def heuristic(graph, line_pool, demand):

    groups = demand_groups(demand)

    shortest_time = {}

    #For every od pair, calc shortest path time
    for origin_demand_pair in demand:
        origin = origin_demand_pair[0]
        destination = origin_demand_pair[1]

        shortest_time[origin_demand_pair] = nx.shortest_path_length(graph, origin, destination, weight="weight")

    selected_lines = []
    remain_lines = list(line_pool.keys())

    total_passengers = sum(demand.values())

    #Start with greedy construction
    while len(selected_lines) < max_lines:

        best_line = None
        best_score = None

        #try every remaining line
        for line_id in remain_lines:

            test_line = selected_lines + [line_id]

            total_cost = 0

            #calculate total cost of selected lines
            for selected_line in test_line:
                line = line_pool[selected_line]
                line_cost = get_line_cost(graph, line)

                total_cost += line_cost

            if total_cost > budget:
                continue

            possible_demand, average_served_time, fair, inequity_score = evaluate(graph, test_line, line_pool, demand, groups, shortest_time)

            #Greedy score here, only save best line
            score = (possible_demand, -inequity_score, -average_served_time)

            #Compares left to right priority
            if best_score is None or score > best_score:
                best_score = score
                best_line = line_id

        if best_line is None:
            break

        selected_lines.append(best_line)
        remain_lines.remove(best_line)

    #Use local search to improve solution
    improved = True
    while improved:
        improved = False

        #Store current solution    
        (old_possible, old_average_time, old_fair, old_inequity_score) = evaluate(graph, selected_lines, line_pool, demand, groups, shortest_time)

        #Try swapping every line, with every combination
        for old_line in selected_lines.copy():
            for new_line in remain_lines.copy():

                test_line = selected_lines.copy()
                test_line.remove(old_line)
                test_line.append(new_line)

                total_cost = 0

                #calculate total cost of selected lines
                for selected_line in test_line:
                    line = line_pool[selected_line]
                    line_cost = get_line_cost(graph, line)

                    total_cost += line_cost

                if total_cost > budget:
                    continue

                (possible_demand, average_served_time, fair, inequity_score) = evaluate(graph, test_line, line_pool, demand, groups, shortest_time)

                #Check if test solution is feasible and better than current solution
                test_isfeasible = (possible_demand == total_passengers and fair and min_lines <= len(test_line) <= max_lines)

                current_isfeasible = (old_possible == total_passengers and old_fair and min_lines <= len(selected_lines) <= max_lines)

                if current_isfeasible:
                    isbetter_sol = (test_isfeasible and average_served_time < old_average_time)

                else:
                    #Use test solution if its better even if infeasible
                    isbetter_sol = (possible_demand > old_possible or (possible_demand == old_possible and inequity_score < old_inequity_score))

                #Swap/Update to new solution if better
                if isbetter_sol:
                    selected_lines.remove(old_line)
                    selected_lines.append(new_line)

                    remain_lines.remove(new_line)
                    remain_lines.append(old_line)

                    improved = True
                    break

            if improved:
                break

    #evaluate final solution
    final_result = evaluate(graph, selected_lines, line_pool, demand, groups, shortest_time)

    possible_demand, average_served_time, fair, inequity_score = final_result

    total_cost = 0

    #calculate total cost of selected lines
    for line_id in selected_lines:
        line = line_pool[line_id]
        line_cost = get_line_cost(graph, line)

        total_cost += line_cost

    #Double check that solution is valid
    valid_solution = (possible_demand == total_passengers and fair and min_lines <= len(selected_lines) <= max_lines and total_cost <= budget)

    return selected_lines, final_result, valid_solution


In [12]:
#Run all Functions and print results
graph, demand = load_data(data)
line_pool = gen_line_pool(graph)
print("Candidate lines:", len(line_pool))

solution, result, valid = heuristic(graph, line_pool, demand)

possible_demand, average_served_time, fair, inequity_score = result

print("Selected lines:")
for line_id in solution:
    print(line_pool[line_id])

print("Passengers served:", possible_demand, "/", sum(demand.values()))
print("Average traveled minutes:", average_served_time)

print("Is it a valid solution?:", valid)

Candidate lines: 290
Selected lines:
[1, 2, 3, 6, 8, 10, 11, 13]
[12, 11, 10, 14, 13]
[5, 4, 6, 15, 7, 10, 11]
[9, 15, 6, 3, 2, 4, 12]
[1, 2, 4, 6, 8, 10, 14]
[1, 2, 3, 6, 8, 15, 7]
Passengers served: 15570.0 / 15570.0
Average traveled minutes: 10.506743737957612
Is it a valid solution?: True
